# Data Preparation

Cleaning steps decided in the initial analysis. The cleaned data is saved to `data/processed/`.

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/DataCoSupplyChainDataset.csv', encoding='cp1252')

## Drop Columns

In [2]:
df = df.drop(columns=['Benefit per order', 'Sales per customer', 'Category Id', 'Customer Id', 'Order Item Product Price', 'Order Item Cardprod Id', 'Product Image', 'Product Description', 'Customer Email', 'Customer Password', 'Product Status', 'Order Zipcode', 'Customer Zipcode', 'Customer Street', 'Latitude', 'Longitude', 'Customer Fname', 'Customer Lname'])

## Remove Cancelled Orders

Cancelled orders (*Order Status* CANCELED or SUSPECTED_FRAUD) were never delivered. Kept in the data, they would count as "not late" in the delivery analysis, and their sales most likely never happened (see the initial analysis).

In [3]:
cancelled = df['Delivery Status'] == 'Shipping canceled'

print(f'Cancelled order items: {cancelled.sum():,} ({cancelled.mean():.1%})')
print(f'Cancelled orders: {df.loc[cancelled, "Order Id"].nunique():,}')

df = df[~cancelled].reset_index(drop=True)

print(f'Remaining order items: {len(df):,}')

Cancelled order items: 7,754 (4.3%)
Cancelled orders: 2,855
Remaining order items: 172,765


## Convert Types

In [4]:
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'])

## Remove Whitespace

In [5]:
columns = ['Category Name', 'Department Name', 'Order Region', 'Product Name']

df[columns] = df[columns].apply(lambda col: col.str.strip())

## Rename Columns

In [6]:
df = df.rename(columns={'order date (DateOrders)': 'order_date', 'shipping date (DateOrders)': 'shipping_date', 'Order Profit Per Order': 'order_item_profit', 'Type': 'payment_type'})

In [7]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
    .str.replace(r'[^\w]', '', regex=True)
)

## Conclusions

### Dropped Columns:

- *Benefit per order*
- *Sales per customer*
- *Category Id*
- *Customer Id*
- *Order Item Product Price*
- *Order Item Cardprod Id*
- *Product Image*
- *Product Description*
- *Customer Email*
- *Customer Password*
- *Product Status*
- *Order Zipcode*
- *Customer Zipcode*
- *Customer Street*
- *Latitude*
- *Longitude*
- *Customer Fname*
- *Customer Lname*

### Removed Rows:

- Cancelled orders (*Delivery Status* = 'Shipping canceled'): 7,754 order items from 2,855 orders. They were never delivered, so they would count as "not late" in the delivery analysis, and their sales most likely never happened.

### Converted Types:

- *order date (DateOrders)* (str -> date)
- *shipping date (DateOrders)* (str -> date)

### Removed Whitespace:

- *Category Name*
- *Department Name*
- *Order Region*
- *Product Name*

### Renamed Columns:

- *order date (DateOrders)* -> *order_date*
- *shipping date (DateOrders)* -> *shipping_date*
- *Order Profit Per Order* -> *order_item_profit*
- *Type* -> *payment_type*

All column names were standardized.

In [8]:
df.to_csv('../data/processed/dataset.csv', index=False)